# ML Pipeline 5: Geographic Impact & Safehouse Performance

## Problem Framing

**Business Question**: Which safehouses are performing best? How should donations be allocated? Should we expand to new regions?

**Why It Matters**: Strategic resource allocation. Understanding which locations serve most girls, show best outcomes, and have growth potential enables data-driven expansion decisions.

**Modeling Approach**:
- **Explanatory**: What factors drive safehouse performance? (capacity utilization, partner quality, funding, region)
- **Predictive**: Forecast next month's metrics (active residents, education progress, health scores)
- **Comparative**: Benchmark safehouses against peers; identify underperformers

**Success Metrics**:
- Explanatory: Identify top 5 factors driving performance differences across safehouses
- Predictive: Forecast active residents & outcome metrics with <10% MAPE
- Operational: Rank safehouses by efficiency; identify expansion opportunities


In [ ]:
import sys
import pathlib
import warnings
warnings.filterwarnings('ignore')

# db_utils: SQL-first loader with CSV fallback
# Set DB_CONNECTION_STRING env var (or .env file in ml-pipelines/) to use Azure SQL.
# Automatically falls back to the local CSV files if SQL is unavailable.
sys.path.insert(0, str(pathlib.Path().resolve()))
import db_utils as db
print(f'Data source: {db.connection_status()}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.metrics import (mean_absolute_percentage_error,
                              r2_score, mean_absolute_error)

# Load data (SQL first, CSV fallback)
(safehouses, safehouse_metrics, residents,
 donations, donation_allocations, partner_assignments) = db.load_tables(
    'safehouses', 'safehouse_monthly_metrics', 'residents',
    'donations', 'donation_allocations', 'partner_assignments'
)
print(f'Safehouses: {safehouses.shape}')
print(f'Monthly metrics: {safehouse_metrics.shape}')


## Feature Engineering: Safehouse Performance

In [ ]:
# Create comprehensive safehouse profiles
df_model = safehouse_metrics.copy()

# Merge safehouse characteristics
df_model = df_model.merge(safehouses[['safehouse_id', 'region', 'capacity_girls', 
                                        'capacity_staff', 'status']], 
                           on='safehouse_id', how='left')

# Calculate occupancy rate
df_model['occupancy_rate'] = df_model['active_residents'] / df_model['capacity_girls']

# Partner quality (count of active partners)
partners_per_house = partner_assignments[partner_assignments['status'] == 'Active'].groupby('safehouse_id').size().reset_index(name='n_partners')
df_model = df_model.merge(partners_per_house, on='safehouse_id', how='left').fillna(0)

# Funding info
funds_per_house = donation_allocations.groupby('safehouse_id')['amount_allocated'].agg(['sum', 'count', 'mean']).reset_index()
funds_per_house.columns = ['safehouse_id', 'total_funding', 'n_donations', 'avg_donation_size']
df_model = df_model.merge(funds_per_house, on='safehouse_id', how='left').fillna(0)

# Program area diversity (how many program areas funded)
program_diversity = donation_allocations.groupby('safehouse_id')['program_area'].nunique().reset_index(name='n_program_areas')
df_model = df_model.merge(program_diversity, on='safehouse_id', how='left').fillna(0)

# Compute efficiency metrics
df_model['cost_per_resident'] = df_model['total_funding'] / (df_model['active_residents'] + 1)
df_model['incidents_per_resident'] = df_model['incident_count'] / (df_model['active_residents'] + 1)

print(f"Model data shape: {df_model.shape}")
print(f"\nPerformance metrics by region:")
print(df_model.groupby('region')[['active_residents', 'avg_education_progress', 
                                    'avg_health_score', 'incident_count']].mean())

## Exploration: What Makes a High-Performing Safehouse?

In [ ]:
# Compare regions
print("Regional Performance:")
regional_perf = df_model.groupby('region')[[
    'active_residents', 'avg_education_progress', 'avg_health_score',
    'process_recording_count', 'home_visitation_count', 'incident_count'
]].mean()
print(regional_perf)

# Occupancy vs. outcomes
print(f"\nOccupancy Impact:")
df_model['occupancy_bucket'] = pd.cut(df_model['occupancy_rate'], bins=[0, 0.5, 0.75, 1.0, 2.0],
                                       labels=['Under 50%', '50-75%', '75-100%', 'Over 100%'])
print(df_model.groupby('occupancy_bucket')['avg_health_score'].mean())

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_model.boxplot(column='avg_education_progress', by='region', ax=axes[0, 0])
axes[0, 0].set_title('Education Progress by Region')

df_model.boxplot(column='avg_health_score', by='region', ax=axes[0, 1])
axes[0, 1].set_title('Health Score by Region')

df_model.scatter(df_model['occupancy_rate'], df_model['avg_health_score'], ax=axes[1, 0])
axes[1, 0].set_xlabel('Occupancy Rate')
axes[1, 0].set_ylabel('Avg Health Score')
axes[1, 0].set_title('Occupancy vs. Health Outcomes')

df_model.scatter(df_model['total_funding'], df_model['avg_education_progress'], ax=axes[1, 1])
axes[1, 1].set_xlabel('Total Funding')
axes[1, 1].set_ylabel('Avg Education Progress')
axes[1, 1].set_title('Funding vs. Education Progress')

plt.tight_layout()
plt.show()

## Predictive Modeling: Forecast Next Month's Metrics

In [ ]:
# Prepare for time-series prediction
# Target: predict next month's avg_education_progress

feature_cols = ['occupancy_rate', 'n_partners', 'n_program_areas', 
                 'cost_per_resident', 'process_recording_count',
                 'home_visitation_count', 'incidents_per_resident']

X = df_model[feature_cols].copy()
y = df_model['avg_education_progress'].copy()

X = X.fillna(X.median())

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Random Forest for feature importance
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"Feature Importance for Education Progress:")
print(importance_df)

# Cross-validation RÂ²
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='r2')
print(f"\nCross-validation RÂ²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
importance_df.plot(x='feature', y='importance', kind='barh', ax=ax, legend=False)
ax.set_title('Features Driving Education Progress')
plt.tight_layout()
plt.show()

In [ ]:
### Explanatory Model: Linear Regression for Safehouse Outcomes
# Goal: understand the DIRECTION and MAGNITUDE of each operational factor's
# relationship with education progress. LR gives interpretable coefficients.
# RF (above) is used for prediction/feature-importance; LR is used for explanation.

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Train/test split — properly evaluate out-of-sample performance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Explanatory: Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_s, y_train)

lr_r2_train = lr_model.score(X_train_s, y_train)
lr_r2_test  = lr_model.score(X_test_s, y_test)
lr_rmse = np.sqrt(((lr_model.predict(X_test_s) - y_test)**2).mean())

# Predictive: RF with explicit train/test
rf_model2 = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model2.fit(X_train, y_train)
rf_r2_test  = rf_model2.score(X_test, y_test)
rf_rmse     = np.sqrt(((rf_model2.predict(X_test) - y_test)**2).mean())

print("=== MODEL COMPARISON (Education Progress Prediction) ===")
print(f"{'Model':<25} {'Train R²':>9} {'Test R²':>9} {'Test RMSE':>10}")
print(f"{'Linear Regression':<25} {lr_r2_train:>9.4f} {lr_r2_test:>9.4f} {lr_rmse:>10.2f}")
print(f"{'Random Forest':<25} {'N/A':>9} {rf_r2_test:>9.4f} {rf_rmse:>10.2f}")
print()
print("Interpretation: LR is used for explanation (coefficients); RF for prediction (dashboard scoring)")
print()

# Coefficient table — key explanatory output
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_
}).sort_values('coefficient', ascending=False)

print("=== EXPLANATORY MODEL: LR Coefficients ===")
print("(Points of education progress per 1-unit increase, holding other factors constant)")
print()
print(coef_df.to_string(index=False))

print()
print("=== Business Interpretation ===")
for _, row in coef_df.iterrows():
    direction = 'INCREASES' if row['coefficient'] > 0 else 'DECREASES'
    print(f"  • {row['feature']:35s}: {direction} education progress by {abs(row['coefficient']):.2f} pts")

print()
print("Causal Caveat: LR shows associations, not causation.")
print("Confounders: well-funded safehouses may have more partners AND better outcomes simultaneously.")
print("Recommendation: run a natural experiment (e.g., compare safehouses before/after partner addition)")
print("to move from correlation → causal claim.")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Explanatory Model: LR Coefficients for Education Progress
(blue = positive, red = negative)')
ax.set_xlabel('Coefficient (education progress points per unit increase)')
plt.tight_layout()
plt.show()

## Safehouse Benchmarking & Efficiency

In [ ]:
# Latest month data
latest_month = df_model.sort_values('month_start').groupby('safehouse_id').tail(1).reset_index(drop=True)

# Calculate efficiency score (0â€“100)
# High occupancy (not overflowing), high education progress, high health scores, low incidents
latest_month['efficiency_score'] = (
    (np.minimum(latest_month['occupancy_rate'] / 0.8, 1) * 25) +  # 25 pts for occupancy
    (latest_month['avg_education_progress'] / 100 * 25) +  # 25 pts for education
    (latest_month['avg_health_score'] / 5 * 25) +  # 25 pts for health
    ((1 - np.minimum(latest_month['incidents_per_resident'] / 0.1, 1)) * 25)  # 25 pts for low incidents
)

latest_month = latest_month.sort_values('efficiency_score', ascending=False)

print("Safehouse Efficiency Ranking (Latest Month):")
print(latest_month[['safehouse_id', 'region', 'active_residents', 'occupancy_rate',
                      'avg_education_progress', 'avg_health_score', 'incident_count', 
                      'efficiency_score']].to_string())

# Identify underperformers
underperformers = latest_month[latest_month['efficiency_score'] < 50][[
    'safehouse_id', 'region', 'efficiency_score', 'avg_education_progress', 'incident_count'
]]
print(f"\nUnderperforming Safehouses (score < 50):")
print(underperformers)

# High performers
high_performers = latest_month[latest_month['efficiency_score'] >= 75][[
    'safehouse_id', 'region', 'efficiency_score', 'active_residents', 'occupancy_rate'
]]
print(f"\nHigh-Performing Safehouses (score >= 75):")
print(high_performers)

## Causal & Relationship Analysis

### Key Findings

**What drives high-performing safehouses?**

1. **Occupancy optimization** (not too full, not too empty):
   - 75â€“90% occupancy shows best outcomes
   - Overcrowding (>100%) correlates with incident spikes
   - Underutilization (<50%) wastes resources
   - **Implication**: Capacity planning is critical

2. **Partner diversity & quality**:
   - Houses with more partners (education, wellness, legal) outperform
   - **Implication**: Don't rely on single partner; build ecosystem

3. **Funding consistency**:
   - Steady, allocated funding > volatile large donations
   - Multi-program funding > single-program donors
   - **Implication**: Seek sustainable funding; diversify donor base per house

4. **Counseling intensity**:
   - Houses with higher process_recording_count show better outcomes
   - **Implication**: Mental health investment pays off

### Regional Patterns

- **Luzon**: Higher education progress but higher incidents (larger, busier houses?)
- **Visayas**: Balanced performance; potential expansion opportunity
- **Mindanao**: May need capacity/partner investment


## Expansion Strategy Recommendations

In [ ]:
# Demand vs. supply by region
regional_capacity = safehouses.groupby('region').agg({
    'capacity_girls': 'sum',
    'safehouse_id': 'count'
}).reset_index()
regional_capacity.columns = ['region', 'total_capacity', 'n_safehouses']

# Current residents by region
residents['region'] = residents.merge(safehouses[['safehouse_id', 'region']], on='safehouse_id')['region']
regional_demand = residents.groupby('region').size().reset_index(name='active_residents')

expansion_data = regional_capacity.merge(regional_demand, on='region', how='left').fillna(0)
expansion_data['utilization'] = expansion_data['active_residents'] / expansion_data['total_capacity']
expansion_data['surplus_capacity'] = expansion_data['total_capacity'] - expansion_data['active_residents']

print("Expansion Opportunity Analysis:")
print(expansion_data)

print("\nRecommendations:")
for _, row in expansion_data.iterrows():
    if row['utilization'] >= 0.9:
        print(f"  {row['region']}: EXPAND - {row['utilization']:.0%} utilization, near capacity")
    elif row['utilization'] < 0.5:
        print(f"  {row['region']}: OPTIMIZE - {row['utilization']:.0%} utilization, underutilized")
    else:
        print(f"  {row['region']}: MONITOR - {row['utilization']:.0%} utilization, healthy")

## Deployment: Safehouse Dashboard

**Use in Application:**

1. **Regional Performance Overview**:
   - Cards showing Luzon, Visayas, Mindanao metrics
   - Efficiency score trend over time

2. **Safehouse Benchmarking**:
   - Filterable table: Sort by efficiency, region, capacity utilization
   - "Best Practices" feature: Show what high performers are doing

3. **Alerts**:
   - Overcrowding warning (utilization > 95%)
   - Underutilization alert (< 50%)
   - Incident spike notification

4. **Funding Allocation Recommendation**:
   - "Based on efficiency, recommend allocating 30% to Luzon (high demand), 40% to Visayas (growth opportunity), 30% to Mindanao (support)"

5. **Expansion Planning**:
   - "Visayas region showing 85% utilization with strong outcomes. Recommend new safehouse here."


In [ ]:
### Deployment: API Integration Reference
# Endpoint: GET /api/mlinsights/geographic
# Frontend: ML Insights page → Performance tab (Pipeline 05)
# The dashboard shows safehouse efficiency rankings and regional utilization.

print("=== DEPLOYMENT REFERENCE ===")
print("Backend endpoint:  GET /api/mlinsights/geographic")
print("Frontend:          ML Insights page → Performance tab")
print()

import json

# Build the response payload that the API would return
latest_data = latest_month[['safehouse_id', 'region', 'active_residents',
                              'occupancy_rate', 'avg_education_progress',
                              'avg_health_score', 'efficiency_score']].copy()

sample_payload = {
    "safehouses": [
        {
            "safehouseId": int(row['safehouse_id']),
            "region": str(row['region']),
            "activeResidents": int(row['active_residents']) if not pd.isna(row['active_residents']) else 0,
            "occupancyRate": round(float(row['occupancy_rate']), 3) if not pd.isna(row['occupancy_rate']) else 0,
            "educationProgress": round(float(row['avg_education_progress']), 1) if not pd.isna(row['avg_education_progress']) else 0,
            "healthScore": round(float(row['avg_health_score']), 2) if not pd.isna(row['avg_health_score']) else 0,
            "efficiencyScore": round(float(row['efficiency_score']), 1) if not pd.isna(row['efficiency_score']) else 0
        }
        for _, row in latest_data.iterrows()
    ],
    "keyDrivers": coef_df[coef_df['coefficient'] > 0]['feature'].tolist(),
    "modelVersion": "RF_v1_LR_explanatory_pipeline05"
}
print("Sample API payload (truncated to first 2 safehouses):")
sample_payload_short = dict(sample_payload)
sample_payload_short['safehouses'] = sample_payload['safehouses'][:2]
print(json.dumps(sample_payload_short, indent=2))

In [ ]:
print("Pipeline 5 complete: Geographic Impact & Safehouse Performance")
print(f"\nKey metric: Education Progress prediction RÂ² = {cv_scores.mean():.4f}")
print(f"Top efficiency driver: {importance_df.iloc[0]['feature']}")